# Homework

In [2]:
import pickle

In [3]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression, Lasso

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow

In [5]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("duration-prediction")

<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1747987569201, experiment_id='1', last_update_time=1747987569201, lifecycle_stage='active', name='duration-prediction', tags={}>

In [8]:
def clean(df):
  # Duration in minutes

  df['duration'] = (df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']) / pd.Timedelta(minutes=1)

  # Remove outliers out of bounds 1 - 60 minutes

  df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]

  return df


### Downloading the data

In [ ]:
df_jan = pd.read_parquet('green_tripdata_2023-01.parquet')
df_jan = clean(df_jan)

### One-hot encoding

In [14]:
def dict_extract(df):
  categorical = ['PULocationID', 'DOLocationID']
  df[categorical] = df[categorical].astype(str)
  df_dict = df[categorical].to_dict(orient='records')

  return df_dict

dict_jan = dict_extract(df_jan)

dv = DictVectorizer()
X_train = dv.fit_transform(dict_jan)

### Training a model

In [15]:
y_train = df_jan['duration'].values
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_train)

In [17]:
with open('models/lin_regression-1.bin', 'wb') as f:
  pickle.dump((dv, lr), f)

In [30]:
with mlflow.start_run() as run:
    mlflow.set_tag("developer", "elvis")
    mlflow.set_tag("model", "Lasso")

    
    alpha = 0.1
    
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    
    

# Q6. Evaluating the model

Now let's apply this model to the validation dataset (February 2023).

What's the RMSE on validation?

In [ ]:
df_feb = pd.read_parquet('green_tripdata_2023-02.parquet')

In [ ]:
df_feb = clean(df_feb)

dict_feb = dict_extract(df_feb)
X_val = dv.transform(dict_feb)

y_val = df_feb['duration'].values

Duration standard deviation 69.28096981389236


In [26]:
y_pred = lr.predict(X_val)
print('RMSE on validation data:', root_mean_squared_error(y_val, y_pred))

RMSE on validation data: 7.356158578732979


In [2]:
import pickle
from sklearn.ensemble import RandomForestRegressor
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("duration-prediction-training")

def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


def run_train():

    X_train, y_train = load_pickle("output/train.pkl")
    X_val, y_val = load_pickle("output/val.pkl")

    mlflow.autolog()
    
        
    depth = 10
    random_state = 0
    
    rf = RandomForestRegressor(max_depth=depth, random_state=random_state)
    
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_pred)


if __name__ == '__main__':
    run_train()


2025/05/24 22:36:18 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/05/24 22:36:18 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '2cba3de0bf9d4d7cb3f9f910e5d8a31b', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


: 

In [ ]:
%tb

In [1]:
import mlflow